# Large Language Models: The Full Tutorial

Welcome to the definitive guide on implementing Large Language Models (LLMs) locally. 

In this tutorial, we will not be using an API (like OpenAI's). Instead, we will download an open-source Transformer model, load its weights into our machine's memory, and run the mathematical forward pass ourselves. We will use **GPT-2**, the grand-daddy of modern LLMs. While much smaller than GPT-4, its architecture and text-generation principles are exactly the same.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing Hugging Face libraries.
2. **Tokenization:** Translating human text into machine-readable integer IDs.
3. **The Forward Pass:** Feeding tensors into the Transformer to get raw logit scores.
4. **Decoding Strategies:** Using math (Greedy vs. Sampling) to pick the next token.

In [ ]:
# Cell 2: Environment Setup and Imports
# To run this locally, you would need: !pip install transformers torch

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch.nn.functional as F

# Set a manual seed for analytical reproducibility
torch.manual_seed(42)

print(f"PyTorch Version: {torch.__version__}")

## Step 1: Loading the Model and Tokenizer

An LLM consists of two tightly coupled components:
1. **The Tokenizer:** A dictionary mapping that chops strings into chunks (tokens) and assigns each chunk a unique integer ID.
2. **The Model (Weights):** The actual neural network that takes a sequence of IDs and predicts the ID of the next token.

We will instantiate both from the Hugging Face hub.

In [ ]:
# Cell 4: Loading GPT-2

model_id = "gpt2" # We use the base 124-million parameter model

# 1. Load the Tokenizer
print("Loading Tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained(model_id)

# 2. Load the Model
print("Loading Model Weights...")
model = GPT2LMHeadModel.from_pretrained(model_id)

# Set model to evaluation mode (turns off dropout layers for deterministic inference)
model.eval()

print("Initialization Complete!")

## Step 2: Tokenization (Human to Machine)

Neural networks cannot process strings. If we want to prompt the model with "Artificial intelligence is", we must tokenize it.

Tokenization does not always split by words. Often, it splits by sub-words (e.g., "unbelievable" -> "un", "believ", "able"). This keeps the vocabulary size manageable and allows the model to handle unseen words analytically.

In [ ]:
# Cell 6: Analyzing the Tokenizer

prompt_text = "Artificial intelligence is"

# Encode the text into PyTorch tensors
# return_tensors='pt' tells it to return PyTorch format
input_ids = tokenizer.encode(prompt_text, return_tensors='pt')

print(f"Human Text: '{prompt_text}'")
print(f"Machine Tensors: {input_ids}")

# Let's analytically prove what each number represents by decoding them one by one
print("\n--- Token Breakdown ---")
for token_id in input_ids[0]:
    decoded_word = tokenizer.decode(token_id)
    print(f"Token ID {token_id.item()} -> '{decoded_word}'")

## Step 3: The Forward Pass (Logits and Softmax)

What happens when we feed these tokens into the model? The model outputs **logits**. Logits are raw, unnormalized scores for every single token in the vocabulary indicating how likely it is to be the *next* token.

GPT-2 has a vocabulary of 50,257 tokens. Therefore, the model will output an array of 50,257 numbers. We apply the **Softmax** function to convert these arbitrary numbers into a clean percentage probability distribution summing to 100%.

In [ ]:
# Cell 8: The Forward Pass

# Disable gradient calculation since we are predicting, not training. Saves memory.
with torch.no_grad():
    outputs = model(input_ids)

# The outputs contain the logits. 
# Shape: [batch_size, sequence_length, vocab_size]
logits = outputs.logits

# We only care about the predictions for the VERY LAST token in our sequence
next_token_logits = logits[0, -1, :] 

print(f"Raw Logits Shape: {next_token_logits.shape}")

# Apply Softmax to convert raw scores to probabilities
probabilities = F.softmax(next_token_logits, dim=-1)

# Find the token with the highest probability (Greedy Search)
best_token_id = torch.argmax(probabilities).item()
best_token_prob = probabilities[best_token_id].item()

best_word = tokenizer.decode(best_token_id)

print(f"\nThe model predicts the next word is: '{best_word}'")
print(f"Confidence score: {best_token_prob * 100:.2f}%")

## Step 4: Autoregressive Generation

Predicting one word is nice, but to generate a paragraph, we must loop this process. 
1. Predict the next token.
2. Append it to the input string.
3. Feed the new, longer string back into the model.
4. Repeat.

Hugging Face provides a `generate()` function that handles this loop in highly optimized C++ under the hood. We will use it here, and explore **Decoding Strategies**. 

If we always pick the highest probability word (**Greedy Search**), LLMs get stuck in boring, repetitive loops. To fix this, we use **Multinomial Sampling** (picking randomly based on probability weights), controlled by **Temperature** (which you saw in the widget above).

In [ ]:
# Cell 10: Controlling Generation Output

base_prompt = "The future of Artificial Intelligence involves"
input_tensor = tokenizer.encode(base_prompt, return_tensors='pt')

print("--- 1. Greedy Generation (Deterministic, often repetitive) ---")
greedy_output = model.generate(
    input_tensor, 
    max_length=30,           # Generate up to 30 tokens total
    no_repeat_ngram_size=2   # Prevent it from repeating the exact same phrases
)
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

print("\n--- 2. Sampling with High Temperature (Creative, potentially chaotic) ---")
creative_output = model.generate(
    input_tensor, 
    max_length=30, 
    do_sample=True,          # Turn on probability sampling
    temperature=1.2,         # Flatten the distribution (more randomness)
    top_k=50                 # Only sample from the top 50 most likely words (safety net)
)
print(tokenizer.decode(creative_output[0], skip_special_tokens=True))

print("\n--- 3. Sampling with Low Temperature (Focused, grounded) ---")
focused_output = model.generate(
    input_tensor, 
    max_length=30, 
    do_sample=True, 
    temperature=0.3,         # Sharpen the distribution (less randomness)
    top_k=50 
)
print(tokenizer.decode(focused_output[0], skip_special_tokens=True))

## Conclusion

You have successfully reverse-engineered how an LLM operates! 

**Summary of the Analytical Workflow:**
1. **Strings to Tensors:** Human text is broken down into a mathematical matrix of Token IDs.
2. **Context Window:** The model looks at the sequence of IDs to establish context.
3. **Logit Projection:** The network outputs a 50,000+ dimensional vector representing the score of every possible next word.
4. **Softmax & Sampling:** The math dictates the next word chosen. Low temperature yields safe, expected text; high temperature yields creative (and sometimes hallucinated) text.
5. **Autoregression:** The chosen word is appended, and the cycle repeats.

This exact mathematical loop is running millions of times a second behind the scenes when you chat with advanced models today!